<a href="https://colab.research.google.com/github/ssk-algoverse/sae-binding/blob/main/circuit/VerySimpleInputPertubation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install transformer_lens

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 4.3 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 192.0/192.0 kB 11.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 739.7/739.7 kB 37.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.4/55.4 kB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.3/18.3 MB 115.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 126.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 94.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 62.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 6.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 12.9 MB/s eta 0:00:00


In [1]:
import torch

In [ ]:
from huggingface_hub import hf_hub_download

REPO_ID = "sebastianhoenig/2L_1H_Entity_Binding"
FILENAME = "2L_1H_Attn_Only.pth"

weights_path = hf_hub_download(repo_id=REPO_ID, filename=FILENAME)

In [3]:
### Model

from transformer_lens import HookedTransformer, HookedTransformerConfig

E = 100 # num entities
A = 100 # num attributes
T = 10 # num types/relations
SEP = E+A+T # as seperator between relations
Q = E+A+T+1 # question token
PAD = E+A+T+2
D_VOCAB = E+A+T+3
IGNORE_INDEX = -100

cfg = HookedTransformerConfig(
    n_layers=2,
    n_heads=1,
    d_model=256,
    d_head=256,
    d_mlp=1024,
    n_ctx=64,
    d_vocab=D_VOCAB,
    act_fn="gelu",
    attn_only=True,
    normalization_type="LN",
)
model = HookedTransformer(cfg)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

# Load the model
# Create a new model instance with the same configuration
pretrained_weights = torch.load(weights_path, map_location=device, weights_only=True)
model.load_state_dict(pretrained_weights)

print("Model loaded successfully.")

Moving model to device:  cuda
Model loaded successfully.


In [5]:
import numpy as np
import pandas as pd
import torch
import ast
from torch.utils.data import Dataset, DataLoader


In [33]:
id_mapping_df = pd.read_csv('id_mapping.csv')
id_to_entity = dict(zip(id_mapping_df['id'], id_mapping_df['name']))

In [9]:
class EntityBindingDataset(Dataset):
    def __init__(self, dataframe, parse_tokens_if_str=True):
        self.df = dataframe.reset_index(drop=True)
        self.parse_tokens_if_str = parse_tokens_if_str

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        seq = row["tokens"]
        tokens = torch.tensor(seq, dtype=torch.long)
        label  = torch.tensor(int(row["label"]), dtype=torch.long)
        return tokens, label

test_df = pd.read_csv('test_df.csv', converters={"tokens": ast.literal_eval})
test_dataset = EntityBindingDataset(test_df)

In [10]:
### Some very basic checks: What happens if we pertube the input sequence

In [34]:
example, label = test_dataset[0]

In [37]:
print("Example tokens and their mapped entities:")
for token_id in example:
    entity_name = id_to_entity.get(token_id.item(), f"Unknown Token {token_id.item()}")
    print(f"Token {token_id.item()} ({entity_name})")

print(f"\nLabel: {label.item()} ({id_to_entity.get(label.item(), f'Unknown Token {label.item()}')})")

Example tokens and their mapped entities:
Token 94 (Bryan)
Token 201 (was born in)
Token 118 (Istanbul)
Token 210 (,)
Token 43 (Keith)
Token 208 (moved to)
Token 105 (Manila)
Token 210 (,)
Token 18 (Lisa)
Token 200 (lives in)
Token 155 (Philadelphia)
Token 210 (,)
Token 95 (Patrick)
Token 204 (studied in)
Token 144 (Riyadh)
Token 210 (,)
Token 48 (Cynthia)
Token 202 (works in)
Token 106 (Shanghai)
Token 210 (,)
Token 37 (Christine)
Token 205 (married in)
Token 111 (Sao Paulo)
Token 210 (,)
Token 54 (Deborah)
Token 206 (visited)
Token 169 (Cape Town)
Token 210 (,)
Token 80 (Anna)
Token 207 (loves)
Token 154 (Miami)
Token 210 (,)
Token 208 (moved to)
Token 43 (Keith)
Token 211 (?)

Label: 105 (Manila)


In [13]:
with torch.no_grad():
  logits = model(example)

In [39]:
pred = logits[0, -1, :].argmax().item()
print(f"Pred: {pred} ({id_to_entity.get(pred, f'Unknown Token {pred}')})")

Pred: 105 (Manila)


In [15]:
top_values, top_indices = torch.topk(logits[0, -1, :], 3)
top_probs = torch.softmax(top_values, dim=0)
for idx, prob in zip(top_indices, top_probs):
    entity_name = id_to_entity.get(idx.item(), f"Unknown Token {idx.item()}")
    print(f"Token {idx.item()} ({entity_name}) → {prob.item():.2%}")

Token 105 (Manila) → 69.50%
Token 106 (Shanghai) → 22.92%
Token 155 (Philadelphia) → 7.58%


In [16]:
### Changing the 105 token

In [40]:
example_2 = example.clone()
example_2[6] = 123
example_2
print("Example with Token 6 perturbation (Manila to Lahore):")
for token_id in example_2:
    entity_name = id_to_entity.get(token_id.item(), f"Unknown Token {token_id.item()}")
    print(f"Token {token_id.item()} ({entity_name})")

print(f"\nLabel: {label.item()} ({id_to_entity.get(label.item(), f'Unknown Token {label.item()}')})")

Example with Token 6 perturbation (Manila to Lahore):
Token 94 (Bryan)
Token 201 (was born in)
Token 118 (Istanbul)
Token 210 (,)
Token 43 (Keith)
Token 208 (moved to)
Token 123 (Lahore)
Token 210 (,)
Token 18 (Lisa)
Token 200 (lives in)
Token 155 (Philadelphia)
Token 210 (,)
Token 95 (Patrick)
Token 204 (studied in)
Token 144 (Riyadh)
Token 210 (,)
Token 48 (Cynthia)
Token 202 (works in)
Token 106 (Shanghai)
Token 210 (,)
Token 37 (Christine)
Token 205 (married in)
Token 111 (Sao Paulo)
Token 210 (,)
Token 54 (Deborah)
Token 206 (visited)
Token 169 (Cape Town)
Token 210 (,)
Token 80 (Anna)
Token 207 (loves)
Token 154 (Miami)
Token 210 (,)
Token 208 (moved to)
Token 43 (Keith)
Token 211 (?)

Label: 105 (Manila)


In [41]:
with torch.no_grad():
  logits_2 = model(example_2)
pred_2 = logits_2[0, -1, :].argmax().item()
print(f"Pred: {pred_2} ({id_to_entity.get(pred_2, f'Unknown Token {pred_2}')})")

Pred: 123 (Lahore)


In [42]:
top_values, top_indices = torch.topk(logits_2[0, -1, :], 3)
top_probs = torch.softmax(top_values, dim=0)
for idx, prob in zip(top_indices, top_probs):
    entity_name = id_to_entity.get(idx.item(), f"Unknown Token {idx.item()}")
    print(f"Token {idx.item()} ({entity_name}) → {prob.item():.2%}")

Token 123 (Lahore) → 93.82%
Token 155 (Philadelphia) → 3.96%
Token 106 (Shanghai) → 2.22%


In [45]:
### Changing another random token
example_3 = example.clone()
example_3[10] = 123
example_3

print("Example with Token 10 perturbation (Philadephia to Lahore):")
for token_id in example_3:
    entity_name = id_to_entity.get(token_id.item(), f"Unknown Token {token_id.item()}")
    print(f"Token {token_id.item()} ({entity_name})")

with torch.no_grad():
  logits_3 = model(example_3)
pred_3 = logits_3[0, -1, :].argmax().item()
print(f"Pred: {pred_3} ({id_to_entity.get(pred_3, f'Unknown Token {pred_3}')})")

Example with Token 10 perturbation (Philadephia to Lahore):
Token 94 (Bryan)
Token 201 (was born in)
Token 118 (Istanbul)
Token 210 (,)
Token 43 (Keith)
Token 208 (moved to)
Token 105 (Manila)
Token 210 (,)
Token 18 (Lisa)
Token 200 (lives in)
Token 123 (Lahore)
Token 210 (,)
Token 95 (Patrick)
Token 204 (studied in)
Token 144 (Riyadh)
Token 210 (,)
Token 48 (Cynthia)
Token 202 (works in)
Token 106 (Shanghai)
Token 210 (,)
Token 37 (Christine)
Token 205 (married in)
Token 111 (Sao Paulo)
Token 210 (,)
Token 54 (Deborah)
Token 206 (visited)
Token 169 (Cape Town)
Token 210 (,)
Token 80 (Anna)
Token 207 (loves)
Token 154 (Miami)
Token 210 (,)
Token 208 (moved to)
Token 43 (Keith)
Token 211 (?)
Pred: 105 (Manila)


In [46]:
top_values, top_indices = torch.topk(logits_3[0, -1, :], 3)
top_probs = torch.softmax(top_values, dim=0)
for idx, prob in zip(top_indices, top_probs):
    entity_name = id_to_entity.get(idx.item(), f"Unknown Token {idx.item()}")
    print(f"Token {idx.item()} ({entity_name}) → {prob.item():.2%}")

Token 105 (Manila) → 79.56%
Token 118 (Istanbul) → 15.87%
Token 106 (Shanghai) → 4.57%


In [22]:
### What happens if we change the entity or attribute - breaking the match

In [52]:
example_4 = example.clone()
example_4[4] = 48 # was 43 before

print("Example with Token 4 perturbation (Keith to Cynthia):")
for token_id in example_4:
    entity_name = id_to_entity.get(token_id.item(), f"Unknown Token {token_id.item()}")
    print(f"Token {token_id.item()} ({entity_name})")

with torch.no_grad():
  logits_4 = model(example_4)
pred_4 = logits_4[0, -1, :].argmax().item()
print(f"Pred: {pred_4} ({id_to_entity.get(pred_4, f'Unknown Token {pred_4}')})")

Example with Token 4 perturbation (Keith to Cynthia):
Token 94 (Bryan)
Token 201 (was born in)
Token 118 (Istanbul)
Token 210 (,)
Token 48 (Cynthia)
Token 208 (moved to)
Token 105 (Manila)
Token 210 (,)
Token 18 (Lisa)
Token 200 (lives in)
Token 155 (Philadelphia)
Token 210 (,)
Token 95 (Patrick)
Token 204 (studied in)
Token 144 (Riyadh)
Token 210 (,)
Token 48 (Cynthia)
Token 202 (works in)
Token 106 (Shanghai)
Token 210 (,)
Token 37 (Christine)
Token 205 (married in)
Token 111 (Sao Paulo)
Token 210 (,)
Token 54 (Deborah)
Token 206 (visited)
Token 169 (Cape Town)
Token 210 (,)
Token 80 (Anna)
Token 207 (loves)
Token 154 (Miami)
Token 210 (,)
Token 208 (moved to)
Token 43 (Keith)
Token 211 (?)
Pred: 106 (Shanghai)


In [53]:
top_values, top_indices = torch.topk(logits_4[0, -1, :], 3)
top_probs = torch.softmax(top_values, dim=0)
for idx, prob in zip(top_indices, top_probs):
    entity_name = id_to_entity.get(idx.item(), f"Unknown Token {idx.item()}")
    print(f"Token {idx.item()} ({entity_name}) → {prob.item():.2%}")

Token 106 (Shanghai) → 54.24%
Token 105 (Manila) → 40.92%
Token 169 (Cape Town) → 4.84%


In [54]:
# 48 (Cynthia) is used in another fact - and it seems that it now predicts both of these attributes - but 106 (Shanghai) with a larger likelihood
# even though the type relation is wrong -> maybe it has a bias towards examples used later in the sequence?

# Lets try editing again - with an entity thats not in the sequence

In [56]:
example_5 = example.clone()
example_5[4] = 5 # was 43 before

print("Example with Token 4 perturbation (Keith to Erica):")
for token_id in example_5:
    entity_name = id_to_entity.get(token_id.item(), f"Unknown Token {token_id.item()}")
    print(f"Token {token_id.item()} ({entity_name})")

with torch.no_grad():
  logits_5 = model(example_5)
pred_5 = logits_5[0, -1, :].argmax().item()
print(f"Pred: {pred_5} ({id_to_entity.get(pred_5, f'Unknown Token {pred_5}')})")

Example with Token 4 perturbation (Keith to Erica):
Token 94 (Bryan)
Token 201 (was born in)
Token 118 (Istanbul)
Token 210 (,)
Token 5 (Erica)
Token 208 (moved to)
Token 105 (Manila)
Token 210 (,)
Token 18 (Lisa)
Token 200 (lives in)
Token 155 (Philadelphia)
Token 210 (,)
Token 95 (Patrick)
Token 204 (studied in)
Token 144 (Riyadh)
Token 210 (,)
Token 48 (Cynthia)
Token 202 (works in)
Token 106 (Shanghai)
Token 210 (,)
Token 37 (Christine)
Token 205 (married in)
Token 111 (Sao Paulo)
Token 210 (,)
Token 54 (Deborah)
Token 206 (visited)
Token 169 (Cape Town)
Token 210 (,)
Token 80 (Anna)
Token 207 (loves)
Token 154 (Miami)
Token 210 (,)
Token 208 (moved to)
Token 43 (Keith)
Token 211 (?)
Pred: 106 (Shanghai)


In [57]:
top_values, top_indices = torch.topk(logits_5[0, -1, :], 3)
top_probs = torch.softmax(top_values, dim=0)
for idx, prob in zip(top_indices, top_probs):
    entity_name = id_to_entity.get(idx.item(), f"Unknown Token {idx.item()}")
    print(f"Token {idx.item()} ({entity_name}) → {prob.item():.2%}")

Token 106 (Shanghai) → 54.36%
Token 105 (Manila) → 41.43%
Token 169 (Cape Town) → 4.21%


In [29]:
## almost the same???  -> discuss
# Now if we change the question entity as well, it should change again

In [64]:
example_6 = example.clone()
example_6[4] = 5 # was 43 (Keith) before
example_6[-2] = 5 # was 43 (Keith) before

print("Example with Token 4 and 33 perturbation (Keith to Erica):")
for token_id in example_6:
    entity_name = id_to_entity.get(token_id.item(), f"Unknown Token {token_id.item()}")
    print(f"Token {token_id.item()} ({entity_name})")

with torch.no_grad():
  logits_6 = model(example_6)
pred_6 = logits_6[0, -1, :].argmax().item()
print(f"Pred: {pred_6} ({id_to_entity.get(pred_6, f'Unknown Token {pred_6}')})")

Example with Token 4 and 33 perturbation (Keith to Erica):
Token 94 (Bryan)
Token 201 (was born in)
Token 118 (Istanbul)
Token 210 (,)
Token 5 (Erica)
Token 208 (moved to)
Token 105 (Manila)
Token 210 (,)
Token 18 (Lisa)
Token 200 (lives in)
Token 155 (Philadelphia)
Token 210 (,)
Token 95 (Patrick)
Token 204 (studied in)
Token 144 (Riyadh)
Token 210 (,)
Token 48 (Cynthia)
Token 202 (works in)
Token 106 (Shanghai)
Token 210 (,)
Token 37 (Christine)
Token 205 (married in)
Token 111 (Sao Paulo)
Token 210 (,)
Token 54 (Deborah)
Token 206 (visited)
Token 169 (Cape Town)
Token 210 (,)
Token 80 (Anna)
Token 207 (loves)
Token 154 (Miami)
Token 210 (,)
Token 208 (moved to)
Token 5 (Erica)
Token 211 (?)
Pred: 105 (Manila)


In [65]:
top_values, top_indices = torch.topk(logits_6[0, -1, :], 3)
top_probs = torch.softmax(top_values, dim=0)
for idx, prob in zip(top_indices, top_probs):
    entity_name = id_to_entity.get(idx.item(), f"Unknown Token {idx.item()}")
    print(f"Token {idx.item()} ({entity_name}) → {prob.item():.2%}")

Token 105 (Manila) → 81.57%
Token 118 (Istanbul) → 11.02%
Token 106 (Shanghai) → 7.42%
